{'es': ['531d53df2e98da67c2123feb45f10ee4-0.gif', '70114013cae7c7fcae7562af169e976a-5.gif', '045853c266c0c61bc7e2072d640ef7ae-4.gif', '3c25b0cb41deff5a42bff8a8cbf5b74b-3.gif', '4198415ea02071537e0a89f9e00dbd57-6.gif', 'd07df66f7f3ad13a75278d3f6ae0170a-4.gif', '9061606fcd3c3a8120aac44293a58957-6.gif', '938fe477a1322f84016bdd66ed791bef-4.gif', '4b06e57bcc131d6f53af2c521c28f9bc-18.gif', 'cf203aaef66c12275b4f667fe4964104-4.gif', '33f220e36b5411b0635843162ff28fac-10.gif', '390fa3161c26460387b7902292d8f111-2.gif', 'c2e4af28922f248116c95a2c422b40cc-3.gif', 'cb94fabb2102f5d0bcb5c18326223863-18.gif', 'f0c8e2a9c2a63e39594eb4a196f97d1c-3.gif', '98633161f2fd1f7aeb8fa53b34a14a54.gif', 'be0f4b0445a5939d1e71423c6067e6f7-11.gif', 'ae1e483e288874e8f3b5afd941ce9042-8.gif', '9b431c37cb11d14769db261ba75df333-6.gif', 'a0025c76df2de63b664e8b2f818e3b4c.gif', '05a6f2b0c952e380564296aa072851f1.gif', '7c9936fd83dda67143fe38c18f20942a-4.gif', '968336f50f602b33d8723c8d294486fc.gif', 'd9493e8bfba00c46c00454bf46a4f

In [33]:
import dataclasses
from typing import List

class OCRResult:
  x1: float
  y1: float
  x2: float
  y2: float
  image_width: int
  image_height: int

  def __init__(self, json_path: str = None):
    if json_path is not None:
      self.from_json_path(json_path)


  def from_json_path(self, json_path: str):
    import json
    with open(json_path, 'r') as f:
      data = json.load(f)
      self.image_width = data['imageWidth']
      self.image_height = data['imageHeight']
      for shape in data['shapes']:
        points = shape['points']
        self.x1 = points[0][0]
        self.y1 = points[0][1]
        self.x2 = points[1][0]
        self.y2 = points[1][1]
        break
  
  def to_dict(self):
    return {
      'x1': self.x1,
      'y1': self.y1,
      'x2': self.x2,
      'y2': self.y2,
      'image_width': self.image_width,
      'image_height': self.image_height
    }

@dataclasses.dataclass
class BannerEntry:
  md5: str
  language: str
  image_path: str
  ocr_path: str
  ocr_result: OCRResult
  html_path: str

  def to_dict(self):
    return {
      'md5': self.md5,
      'language': self.language,
      'image_path': self.image_path,
      'ocr_path': self.ocr_path,
      'ocr_result': self.ocr_result.to_dict(),
      'html_path': self.html_path
    }

In [34]:
# find all the .gif files under a directory and print them

import os

LANGUAGES = ['es',"en","zh",'ja']

dir = './data'

language_dict = {}

def find_html_path(md5: str, language: str):
  for root, dirs, files in os.walk(dir + '/' +"html_" +language):
    for file in files:
      if file.endswith('.html') and md5 in file:
        return os.path.join(root, file)

def find_json_path(md5: str, language: str):
  for root, dirs, files in os.walk(dir + '/' + language):
    for file in files:
      if file.endswith('.json') and md5 in file:
        return os.path.join(root, file)
      

for language in LANGUAGES:
  for root, dirs, files in os.walk(dir + '/' + language):
    for file in files:
      if file.endswith('.gif'):
        md5 = file.split('-')[0]
        image_path = os.path.join(root, file)
        ocr_path = find_json_path(md5, language)
        html_path = find_html_path(md5, language)

        if ocr_path is None or html_path is None:
          continue
        ocr_result = OCRResult(ocr_path)
        entry = BannerEntry(md5, language, image_path, ocr_path, ocr_result, html_path)

        if language not in language_dict:
          language_dict[language] = []
        
        language_dict[language].append(entry)


In [ ]:
dict_list = []
for language, entries in language_dict.items():
  for entry in entries:
    dict_list.append(entry.to_dict())


import json
with open('banner_data.json', 'w') as f:
  json.dump(dict_list, f, indent=2)